In [3]:
"""
Dataset Statistics Analysis for Machine Learning Paper
Provides comprehensive statistics about the multimodal gesture dataset
"""

import os
import sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

from datautils.midas import MultimodalGestureDataset

In [ ]:


def compute_dataset_statistics(dataset: MultimodalGestureDataset, 
                               name: str = "Dataset") -> Dict:
    """
    Compute comprehensive statistics about the dataset.
    
    Returns:
        Dictionary containing various statistics
    """
    stats = {
        'name': name,
        'total_samples': len(dataset),
        'num_classes': dataset.num_classes,
        'classes': dataset.classes,
        'class_map': dataset.class_map,
    }
    
    # Per-class window counts
    class_counts = defaultdict(int)
    class_durations = defaultdict(list)  # in frames
    class_durations_seconds = defaultdict(list)
    
    for sample in dataset.samples:
        gesture = sample['gesture_code']
        duration_frames = sample['end'] - sample['start']
        duration_seconds = duration_frames / dataset.sample_rate
        
        class_counts[gesture] += 1
        class_durations[gesture].append(duration_frames)
        class_durations_seconds[gesture].append(duration_seconds)
    
    stats['class_counts'] = dict(class_counts)
    stats['class_durations_frames'] = {k: np.array(v) for k, v in class_durations.items()}
    stats['class_durations_seconds'] = {k: np.array(v) for k, v in class_durations_seconds.items()}
    
    # Compute duration statistics per class
    duration_stats = {}
    for gesture in dataset.classes:
        durations_sec = class_durations_seconds[gesture]
        duration_stats[gesture] = {
            'count': len(durations_sec),
            'mean_sec': np.mean(durations_sec),
            'std_sec': np.std(durations_sec),
            'min_sec': np.min(durations_sec),
            'max_sec': np.max(durations_sec),
            'median_sec': np.median(durations_sec),
            'q25_sec': np.percentile(durations_sec, 25),
            'q75_sec': np.percentile(durations_sec, 75),
        }
    
    stats['duration_stats'] = duration_stats
    
    # Overall statistics
    all_durations = np.concatenate([v for v in class_durations_seconds.values()])
    stats['overall'] = {
        'total_windows': len(dataset),
        'mean_duration_sec': np.mean(all_durations),
        'std_duration_sec': np.std(all_durations),
        'min_duration_sec': np.min(all_durations),
        'max_duration_sec': np.max(all_durations),
        'median_duration_sec': np.median(all_durations),
        'total_duration_sec': np.sum(all_durations),
        'total_duration_min': np.sum(all_durations) / 60,
    }
    
    # Trial-level statistics
    trials = set()
    for sample in dataset.samples:
        start_idx = sample['start']
        trial_id = dataset.df.loc[start_idx, 'trial_id']
        trials.add(trial_id)
    
    stats['num_trials'] = len(trials)
    stats['trial_ids'] = sorted(trials)
    
    # Modality information
    stats['modalities'] = list(dataset.modality_cols.keys())
    stats['modality_dims'] = {mod: len(cols) for mod, cols in dataset.modality_cols.items()}
    
    return stats


def print_statistics_report(stats: Dict):
    """
    Print a formatted statistics report suitable for a paper.
    """
    print("=" * 80)
    print(f"DATASET STATISTICS REPORT: {stats['name']}")
    print("=" * 80)
    
    print("\n### OVERALL STATISTICS ###")
    print(f"Total number of samples (windows): {stats['total_samples']:,}")
    print(f"Number of gesture classes: {stats['num_classes']}")
    print(f"Number of trials: {stats['num_trials']}")
    print(f"Gesture classes: {', '.join(stats['classes'])}")
    
    overall = stats['overall']
    print(f"\nTotal dataset duration: {overall['total_duration_min']:.2f} minutes ({overall['total_duration_sec']:.2f} seconds)")
    print(f"Mean window duration: {overall['mean_duration_sec']:.3f} ± {overall['std_duration_sec']:.3f} seconds")
    print(f"Median window duration: {overall['median_duration_sec']:.3f} seconds")
    print(f"Duration range: [{overall['min_duration_sec']:.3f}, {overall['max_duration_sec']:.3f}] seconds")
    
    print("\n### MODALITY INFORMATION ###")
    print(f"Available modalities: {', '.join(stats['modalities'])}")
    for mod, dim in stats['modality_dims'].items():
        print(f"  - {mod}: {dim} features")
    
    print("\n### PER-CLASS STATISTICS ###")
    print(f"{'Class':<20} {'Count':<8} {'Mean(s)':<10} {'Std(s)':<10} {'Min(s)':<10} {'Max(s)':<10} {'Median(s)':<10}")
    print("-" * 88)
    
    for gesture in stats['classes']:
        ds = stats['duration_stats'][gesture]
        print(f"{gesture:<20} {ds['count']:<8} {ds['mean_sec']:<10.3f} {ds['std_sec']:<10.3f} "
              f"{ds['min_sec']:<10.3f} {ds['max_sec']:<10.3f} {ds['median_sec']:<10.3f}")
    
    print("\n### CLASS DISTRIBUTION ###")
    total = sum(stats['class_counts'].values())
    print(f"{'Class':<20} {'Count':<8} {'Percentage':<12}")
    print("-" * 40)
    for gesture in stats['classes']:
        count = stats['class_counts'][gesture]
        pct = (count / total) * 100
        print(f"{gesture:<20} {count:<8} {pct:>10.2f}%")
    
    # Class balance metrics
    counts = np.array([stats['class_counts'][g] for g in stats['classes']])
    imbalance_ratio = counts.max() / counts.min()
    print(f"\nClass imbalance ratio (max/min): {imbalance_ratio:.2f}")
    print(f"Coefficient of variation: {np.std(counts) / np.mean(counts):.3f}")
    
    print("\n" + "=" * 80)


def create_visualization_plots(stats: Dict, save_dir: str = "./dataset_analysis"):
    """
    Create visualization plots for the dataset statistics.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Set style
    sns.set_style("whitegrid")
    sns.reset_orig()
    
    plt.rcParams['figure.dpi'] = 300
    
    classes = stats['classes']
    
    # 1. Class distribution bar plot
    fig, ax = plt.subplots(figsize=(10, 6))
    counts = [stats['class_counts'][g] for g in classes]
    bars = ax.bar(classes, counts, color='steelblue', alpha=0.8, edgecolor='black')
    ax.set_xlabel('Gesture Class', fontsize=12, )
    ax.set_ylabel('Number of Samples', fontsize=12, )
    ax.set_title('Class Distribution', fontsize=14, )
    ax.tick_params(axis='x', rotation=45)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/class_distribution.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/class_distribution.png")
    
    # 2. Duration distributions box plot
    fig, ax = plt.subplots(figsize=(12, 6))
    duration_data = [stats['class_durations_seconds'][g] for g in classes]
    bp = ax.boxplot(duration_data, labels=classes, patch_artist=True,
                     showmeans=True, meanline=True)
    
    # Color boxes
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax.set_xlabel('Gesture Class', fontsize=12, )
    ax.set_ylabel('Duration (seconds)', fontsize=12, )
    ax.set_title('Duration Distribution per Class', fontsize=14, )
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/duration_distributions.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/duration_distributions.png")
    
    # 3. Combined plot: count and mean duration
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: counts
    ax1.bar(classes, counts, color='steelblue', alpha=0.8, edgecolor='black')
    ax1.set_xlabel('Gesture Class', fontsize=11, )
    ax1.set_ylabel('Number of Samples', fontsize=11, )
    ax1.set_title('Sample Count per Class', fontsize=12, )
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Right: mean durations with error bars
    means = [stats['duration_stats'][g]['mean_sec'] for g in classes]
    stds = [stats['duration_stats'][g]['std_sec'] for g in classes]
    ax2.bar(classes, means, yerr=stds, color='coral', alpha=0.8, 
            edgecolor='black', capsize=5)
    ax2.set_xlabel('Gesture Class', fontsize=11, )
    ax2.set_ylabel('Mean Duration (seconds)', fontsize=11, )
    ax2.set_title('Mean Duration per Class (±1 std)', fontsize=12, )
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/combined_stats.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/combined_stats.png")
    
    # 4. Histogram of all durations
    fig, ax = plt.subplots(figsize=(10, 6))
    all_durations = np.concatenate([stats['class_durations_seconds'][g] for g in classes])
    ax.hist(all_durations, bins=30, color='seagreen', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(all_durations), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(all_durations):.2f}s')
    ax.axvline(np.median(all_durations), color='blue', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(all_durations):.2f}s')
    ax.set_xlabel('Duration (seconds)', fontsize=12, )
    ax.set_ylabel('Frequency', fontsize=12, )
    ax.set_title('Overall Duration Distribution', fontsize=14, )
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/duration_histogram.png", bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_dir}/duration_histogram.png")


def export_stats_to_csv(stats: Dict, save_dir: str = "./dataset_analysis"):
    """
    Export statistics to CSV files for easy inclusion in papers/reports.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Per-class statistics table
    rows = []
    for gesture in stats['classes']:
        ds = stats['duration_stats'][gesture]
        rows.append({
            'Class': gesture,
            'Count': ds['count'],
            'Mean_Duration_sec': ds['mean_sec'],
            'Std_Duration_sec': ds['std_sec'],
            'Min_Duration_sec': ds['min_sec'],
            'Max_Duration_sec': ds['max_sec'],
            'Median_Duration_sec': ds['median_sec'],
            'Q25_Duration_sec': ds['q25_sec'],
            'Q75_Duration_sec': ds['q75_sec'],
            'Percentage': (ds['count'] / stats['total_samples']) * 100,
        })
    
    df = pd.DataFrame(rows)
    csv_path = f"{save_dir}/class_statistics.csv"
    df.to_csv(csv_path, index=False, float_format='%.3f')
    print(f"Saved: {csv_path}")
    
    # Overall statistics
    overall_df = pd.DataFrame([stats['overall']])
    overall_csv = f"{save_dir}/overall_statistics.csv"
    overall_df.to_csv(overall_csv, index=False, float_format='%.3f')
    print(f"Saved: {overall_csv}")


def main():
    """
    Main analysis script
    """
    # Configuration
    ROOT_DIR = "/standard/UVA-DSA/MIDAS/Organized/final_data"
    
    # List your CSV files here
    csv_files = [
        f"{ROOT_DIR}/t1/synched_data/final_annotation_t1.csv",
        f"{ROOT_DIR}/t2/synched_data/final_annotation_t2.csv",
        f"{ROOT_DIR}/t3/synched_data/final_annotation_t3.csv",
        f"{ROOT_DIR}/t4/synched_data/final_annotation_t4.csv",
        f"{ROOT_DIR}/t5/synched_data/final_annotation_t5.csv",
        f"{ROOT_DIR}/t6/synched_data/final_annotation_t6.csv",
        f"{ROOT_DIR}/t7/synched_data/final_annotation_t7.csv"
        # Add more as needed
    ]
    
    print("Loading dataset...")
    
    # Create dataset with your desired configuration
    dataset = MultimodalGestureDataset(
        csv_paths=csv_files,
        clip_len=-1,              # Your windowing configuration
        step=1,                   # Your step size
        sample_rate=10,           # Your sampling rate
        ignore_clutch=True,
        clutch_pressed_value=0,
        
        include_modalities=["trakstar", "sw_left", "sw_right", "console"],
        modality_selections={
            "trakstar": [
                "trakstar_sensor_0_*x", "trakstar_sensor_0_*y", "trakstar_sensor_0_azimuth",
                "trakstar_sensor_2_*x", "trakstar_sensor_2_*y", "trakstar_sensor_2_azimuth",
            ],
        },
        normalize=True,
    )
    
    print(f"Dataset loaded: {len(dataset)} samples\n")
    
    # Compute statistics
    print("Computing statistics...")
    stats = compute_dataset_statistics(dataset, name="Multimodal Gesture Dataset")
    
    # Print report
    print_statistics_report(stats)
    
    # Create visualizations
    print("\nCreating visualizations...")
    save_dir = "./dataset_analysis"
    create_visualization_plots(stats, save_dir=save_dir)
    
    # Export to CSV
    print("\nExporting statistics to CSV...")
    export_stats_to_csv(stats, save_dir=save_dir)
    
    print(f"\n✓ Analysis complete! Results saved to: {save_dir}/")
    print("\nFiles generated:")
    print("  - class_statistics.csv")
    print("  - overall_statistics.csv")
    print("  - class_distribution.png")
    print("  - duration_distributions.png")
    print("  - combined_stats.png")
    print("  - duration_histogram.png")


if __name__ == "__main__":
    main()

Loading dataset...
Total rows with -1 in checked columns: 1181 / 11238
Dropped 1181 rows containing -1 anywhere.
After dropping rows: 10057
Total rows with -1 in checked columns: 783 / 9867
Dropped 783 rows containing -1 anywhere.
After dropping rows: 9084
Total rows with -1 in checked columns: 2589 / 13209
Dropped 2589 rows containing -1 anywhere.
After dropping rows: 10620
Total rows with -1 in checked columns: 2144 / 24963
Dropped 2144 rows containing -1 anywhere.
After dropping rows: 22819
Total rows with -1 in checked columns: 1582 / 6705
Dropped 1582 rows containing -1 anywhere.
After dropping rows: 5123
Total rows with -1 in checked columns: 1216 / 21065
Dropped 1216 rows containing -1 anywhere.
After dropping rows: 19849
Total rows with -1 in checked columns: 2483 / 39517
Dropped 2483 rows containing -1 anywhere.
After dropping rows: 37034
Dataset will load image features: []
Dataset loaded: 345 samples

Computing statistics...
DATASET STATISTICS REPORT: Multimodal Gesture Data

/tmp/ipykernel_49308/2833401700.py:170: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(duration_data, labels=classes, patch_artist=True,


Saved: ./dataset_analysis/duration_distributions.png
Saved: ./dataset_analysis/combined_stats.png
Saved: ./dataset_analysis/duration_histogram.png

Exporting statistics to CSV...
Saved: ./dataset_analysis/class_statistics.csv
Saved: ./dataset_analysis/overall_statistics.csv

✓ Analysis complete! Results saved to: ./dataset_analysis/

Files generated:
  - class_statistics.csv
  - overall_statistics.csv
  - class_distribution.png
  - duration_distributions.png
  - combined_stats.png
  - duration_histogram.png
